#### Using Proto-ML [1] to analyse the drug reviews dataset [2]

[1] https://github.com/yx131/proto-lm/tree/main

[2] https://www.kaggle.com/datasets/mohamedabdelwahabali/drugreview/data

### Global imports

In [1]:
import argparse
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from ProtoLM import proto_lm
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
import datasets

### Global Params

In [2]:
train_the_model = True

In [3]:
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import os
import nltk
nltk.download('vader_lexicon')

def add_sentiment_features(df, text_column):
    """
    Adds sentiment analysis features to the dataset using VADER.

    Args:
        df (pd.DataFrame): The input DataFrame containing the text data.
        text_column (str): The name of the column containing the text.

    Returns:
        pd.DataFrame: The updated DataFrame with sentiment features.
    """
    # Initialize the VADER sentiment analyzer
    analyzer = SentimentIntensityAnalyzer()

    # Compute sentiment scores for each text
    sentiment_scores = df[text_column].fillna("").apply(analyzer.polarity_scores)

    # Convert the sentiment scores into a DataFrame
    sentiment_df = pd.json_normalize(sentiment_scores)

    # Rename the sentiment columns for clarity
    sentiment_df.rename(columns={
        'neg': 'sentiment_neg',
        'neu': 'sentiment_neu',
        'pos': 'sentiment_pos',
        'compound': 'sentiment_compound'
    }, inplace=True)

    # Join the sentiment features with the original DataFrame
    df = df.join(sentiment_df.set_index(df.index))
    return df

# Process all datasets
data_files = {
    "train": "../Data/drug_review_train_clean.csv",
    "validation": "../Data/drug_review_validation_clean.csv",
    "test": "../Data/drug_review_test_clean.csv"
}

for split, file_path in data_files.items():
    output_path = file_path.replace("_clean.csv", "_with_sentiment.csv")
    
    # Check if the output file already exists
    if os.path.exists(output_path):
        print(f"Skipping {split} dataset as {output_path} already exists.")
        continue

    print(f"Processing {split} dataset...")
    
    # Load the dataset
    data = pd.read_csv(file_path)
    
    # Add sentiment features
    data = add_sentiment_features(data, text_column="review_clean")
    
    # Save the updated dataset
    data.to_csv(output_path, index=False)
    print(f"Sentiment features added and saved to {output_path}")

Skipping train dataset as ../Data/drug_review_train_with_sentiment.csv already exists.
Skipping validation dataset as ../Data/drug_review_validation_with_sentiment.csv already exists.
Skipping test dataset as ../Data/drug_review_test_with_sentiment.csv already exists.


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\bar24\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Drug Review Data Class

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd

class sst_datamodule(pl.LightningDataModule):
    loader_columns = [
        "datasets_idx",
        "input_ids",
        "token_type_ids",
        "attention_mask",
        "start_positions",
        "end_positions",
        "labels",
        "sentiment_features",
        "sentiment_neg",
        "sentiment_neu",
        "sentiment_pos",
        "sentiment_compound"
    ]

    def __init__(
            self,
            model_name_or_path: str,
            max_seq_length: int = 128,
            train_batch_size: int = 32,
            eval_batch_size: int = 32,
            dataset=None,  # Accept pre-loaded dataset
            **kwargs,
    ):
        super().__init__()
        self.model_name_or_path = model_name_or_path
        self.max_seq_length = max_seq_length
        self.train_batch_size = train_batch_size
        self.eval_batch_size = eval_batch_size
        self.dataset = dataset

        self.text_fields = ['review_clean']
        self.num_labels = 10  # Number of classes in drug review dataset, a user can rate from 1 to 10
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name_or_path, use_fast=True)

    def load_dataset_locally(self):
        from datasets import load_dataset
        data_files = {
            "train": "../Data/drug_review_train_with_sentiment.csv",
            "validation": "../Data/drug_review_validation_with_sentiment.csv",
            "test": "../Data/drug_review_test_with_sentiment.csv"
        }
        self.dataset = load_dataset("csv", data_files=data_files)

    def setup(self, stage: str = None):
        if self.dataset is None:
            print("Loading dataset from local files...")
            self.load_dataset_locally()

        for split in self.dataset.keys():
            print(f'split is: {split}')
            

            if "rating" in self.dataset[split].column_names:
                self.dataset[split] = self.dataset[split].rename_column("rating", "label")


            self.dataset[split] = self.dataset[split].map(
                self.convert_to_features,
                batched=True,
                # remove_columns=["label", "Unnamed: 0"],
            )
            
            self.columns = [c for c in self.dataset[split].column_names if c in self.loader_columns]
            print(f'self.columns: {self.columns}')
            self.dataset[split].set_format(type="torch", columns=self.columns)

        self.eval_splits = [x for x in self.dataset.keys() if "validation" in x]

    def train_dataloader(self):
        print(f'returned self batch size: {self.train_batch_size}')
        return DataLoader(self.dataset["train"], batch_size=self.train_batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.dataset["validation"], batch_size=self.eval_batch_size)

    def test_dataloader(self):
        return DataLoader(self.dataset["test"], batch_size=self.eval_batch_size)

    def convert_to_features(self, example_batch, indices=None):
        if len(self.text_fields) > 1:
            texts_or_text_pairs = list(zip(example_batch[self.text_fields[0]], example_batch[self.text_fields[1]]))
        else:
            texts_or_text_pairs = example_batch[self.text_fields[0]]

        features = self.tokenizer.batch_encode_plus(
            texts_or_text_pairs,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=self.max_seq_length
        )

        # features["labels"] = [1 if int(label) >= 7 else 0 for label in example_batch["label"]]
        # Convert labels to zero-indexed (1-10 becomes 0-9)
        features["labels"] = [int(label) - 1 for label in example_batch["label"]]
        # # Add sentiment features
        # features["sentiment_neg"] = torch.tensor(example_batch["sentiment_neg"]).unsqueeze(1)
        # features["sentiment_neu"] = torch.tensor(example_batch["sentiment_neu"]).unsqueeze(1)
        # features["sentiment_pos"] = torch.tensor(example_batch["sentiment_pos"]).unsqueeze(1)
        # features["sentiment_compound"] = torch.tensor(example_batch["sentiment_compound"]).unsqueeze(1)
        # Combine sentiment features into a single tensor
        sentiment_features = torch.stack([
            torch.tensor(example_batch["sentiment_neg"]),
            torch.tensor(example_batch["sentiment_neu"]),
            torch.tensor(example_batch["sentiment_pos"]),
            torch.tensor(example_batch["sentiment_compound"])
        ], dim=1)  # Shape: (batch_size, 4)
        features["sentiment_features"] = sentiment_features
        return features

In [5]:
from transformers import AutoConfig

# Parameters for Proto-LM training on the drug review dataset
model_name = 'bert-base-uncased'  # Backbone LLM model to load
# Maximum sentence length to pad/truncate to
args = {
    'model_name': model_name,        # backbone LLM model to load
    'max_seq_length': 100,                # maximum sentence length to pad/truncate to
    'num_prototypes': 100,               # number of prototypes to train
    'hidden_shape': 1024,                # hidden shape of each prototype, should match LLM output
    'num_classes': 10,                    # number of output classes
    'cohsep_ratio': 0.5,                 # ratio of prototypes in class to push/pull
    'lambda0': 0.5,                      # lambda0 in loss
    'lr': 3e-4,                          # initial learning rate
    'proto_training_weights': 1,         # whether to train prototype weights (1=True, 0=False)
    'batch_size': 64,                   # batch size for dataloader
    'logger_dir': 'tb_logs',             # directory for the logger to store training details
    'checkpoint_dir': 'ckpt_dir',        # directory to store checkpoints
    'config_subdir': 'config_subdir',    # subdirectory for checkpoints of a certain config
    'max_epochs': 1,                     # number of epochs to train
    'num_gpu': 1,                        # number of gpus to train on
    'load_model': model_name,  # path to load a pretrained model, if any
}



# Dynamically fetch hidden size from the model configuration
config = AutoConfig.from_pretrained(args['model_name'])
hidden_size = config.hidden_size  # Dynamically get the hidden size (768 for bert-base-uncased)
args['hidden_shape'] = hidden_size

print(f'args: {args}')


# get data module
drug_review_dm = sst_datamodule(
    model_name_or_path=args['model_name'],
    max_seq_length=args['max_seq_length'],
    train_batch_size=args['batch_size'],
    eval_batch_size=args['batch_size']
)
drug_review_dm.setup(stage='fit')

args: {'model_name': 'bert-base-uncased', 'max_seq_length': 100, 'num_prototypes': 100, 'hidden_shape': 768, 'num_classes': 10, 'cohsep_ratio': 0.5, 'lambda0': 0.5, 'lr': 0.0003, 'proto_training_weights': 1, 'batch_size': 64, 'logger_dir': 'tb_logs', 'checkpoint_dir': 'ckpt_dir', 'config_subdir': 'config_subdir', 'max_epochs': 1, 'num_gpu': 1, 'load_model': 'bert-base-uncased'}
Loading dataset from local files...
split is: train
self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']
split is: validation


Map:   0%|          | 0/27703 [00:00<?, ? examples/s]

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']
split is: test


Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [6]:

print(f"loading a model: {args['load_model']}")

base_model = AutoModelForSequenceClassification.from_pretrained(
    args['model_name'], ignore_mismatched_sizes=True
)

if hasattr(base_model, "roberta"):
    llm_model = base_model.roberta
elif hasattr(base_model, "bert"):
    llm_model = base_model.bert
else:
    llm_model = base_model

proto = proto_lm(
    pretrained_model=llm_model,
    max_seq_length=args['max_seq_length'],
    num_prototypes=args['num_prototypes'],
    hidden_shape=args['hidden_shape'],
    num_classes=args['num_classes'],
    cohsep_ratio=args['cohsep_ratio'],
    lambda0=args['lambda0'],
    lr=args['lr'],
    proto_training_weights=bool(args['proto_training_weights']),
)

loading a model: bert-base-uncased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Train

In [7]:
if train_the_model:
    # get training utilities like logger and checkpoints
    from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
    from pytorch_lightning.callbacks import ModelCheckpoint

    tb_logger = TensorBoardLogger(f"{args['logger_dir']}", name="drug_review_tensorboard_logs")
    ckpt_path = f"{args['checkpoint_dir']}/{args['config_subdir']}"
    checkpoint_callback = ModelCheckpoint(
        dirpath=ckpt_path,
        monitor='val_loss',
        save_top_k=3,
        filename="{epoch}-{val_loss:.4f}-{val_accuracy:.4f}"
    )

    # get trainer object
    trainer = pl.Trainer(
        max_epochs=args['max_epochs'],
        accelerator="auto",
        devices=1,  # or just remove this line for auto
        logger=tb_logger,
        callbacks=[checkpoint_callback],
        log_every_n_steps=50  # Log every 50 steps instead of every step
    )

    trainer.fit(proto, datamodule=drug_review_dm)

    # Optionally test or save misclassified
    # trainer.test(proto, datamodule=drug_review_dm, ckpt_path=args['load_model'])
    # torch.save(proto.misclassified, 'drug_review_logs/misclassed.pt')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


split is: train


Map:   0%|          | 0/110811 [00:00<?, ? examples/s]

C:\Users\bar24\AppData\Local\Temp\ipykernel_20740\602209391.py:107: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(example_batch["sentiment_neg"]),
C:\Users\bar24\AppData\Local\Temp\ipykernel_20740\602209391.py:108: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(example_batch["sentiment_neu"]),
C:\Users\bar24\AppData\Local\Temp\ipykernel_20740\602209391.py:109: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(example_batch["sentiment_pos"]),
C:\Users\bar24\AppData\Local\Temp\ipykernel_20740\602209391.py:1

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']
split is: validation


...


  | Name          | Type      | Params | Mode 
----------------------------------------------------
0 | LLM           | BertModel | 109 M  | eval 
1 | fc_word_level | Linear    | 590 K  | train
2 | dense         | Linear    | 1.1 K  | train
  | other params  | n/a       | 77.6 K | n/a  
----------------------------------------------------
110 M     Trainable params
0         Non-trainable params
110 M     Total params
440.606   Total estimated model params size (MB)
2         Modules in train mode
228       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


similarities shape: torch.Size([64, 100])
combined_features shape: torch.Size([64, 100])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x100 and 104x10)

### Save the trained model

In [ ]:
if train_the_model:
# Save the final model weights
    torch.save(proto.state_dict(), "final_proto_model.pt")
    print("Training complete. Model weights saved as 'final_proto_model.pt'.")

### Load the trained model

In [ ]:
model = proto.load_state_dict(torch.load("final_proto_model.pt", weights_only=False))
print("Model loaded successfully from 'final_proto_model.pt'.")

### Evalute the model

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
from tqdm import tqdm  # Import tqdm for the loading bar

# Function to calculate MSE, MAE, and RMSE
def calculate_metrics(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing Batches", leave=False):  # Add tqdm for progress bar
            # Move data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Pass only input_ids and attention_mask to the model
            # outputs = model(
            #     input_ids=input_ids,
            #     attention_mask=attention_mask,
            #     sentiment_features=None  # Exclude sentiment features
            # )
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                sentiment_features=torch.cat((
                    batch["sentiment_neg"],
                    batch["sentiment_neu"],
                    batch["sentiment_pos"],
                    batch["sentiment_compound"]
                ), dim=1)
            )
            preds = torch.argmax(outputs["logits"], dim=1)  # Predicted class

            # Collect predictions and labels
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Flatten predictions and labels
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Calculate metrics
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    rmse = np.sqrt(mse)

    return mse, mae, rmse

# Load the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proto.to(device)

# Calculate metrics for train, validation, and test sets
datasets = {
    "Train": drug_review_dm.train_dataloader(),
    "Validation": drug_review_dm.val_dataloader(),
    "Test": drug_review_dm.test_dataloader()
}

results = []

for dataset_name, dataloader in datasets.items():
    print(f"Calculating metrics for {dataset_name}...")
    mse, mae, rmse = calculate_metrics(proto, dataloader, device)
    print(f"{dataset_name} Metrics:")
    print(f"  MSE: {mse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    results.append({"Dataset": dataset_name, "MSE": mse, "MAE": mae, "RMSE": rmse})

# Export results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv("metrics_results.csv", index=False)
print("Metrics saved to 'metrics_results.csv'")

### Calc Quantus Metrics

In [ ]:
import quantus
import torch
import numpy as np
import pandas as pd

# Define your model and data
model = proto  # Your Proto-LM model
model_name_for_csv = f"ProtoLM_{model_name}"  # Add your model name here
model.eval()  # Set the model to evaluation mode

# Define a wrapper for your model to work with Quantus
class ModelWrapper:
    def __init__(self, model):
        self.model = model

    def __call__(self, input_ids, attention_mask, sentiment_features=None):
        with torch.no_grad():
            # Ensure input_ids is of type LongTensor
            input_ids = input_ids.long()

            # Call the proto model's forward method
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                sentiment_features=sentiment_features
            )
            logits = outputs['logits']  # Extract logits from the output dictionary
        return logits

wrapped_model = ModelWrapper(model)

# Define a sample batch of data (input_ids, attention_mask, sentiment_features)
batch = next(iter(drug_review_dm.test_dataloader()))
input_ids = batch["input_ids"].long()  # Ensure input_ids is LongTensor
attention_mask = batch["attention_mask"]
sentiment_features = torch.cat((
    batch["sentiment_neg"],
    batch["sentiment_neu"],
    batch["sentiment_pos"],
    batch["sentiment_compound"]
), dim=1)
labels = batch["labels"]

# Define an attribution method (e.g., Integrated Gradients)
from captum.attr import IntegratedGradients
ig = IntegratedGradients(wrapped_model)

# Generate attributions for the input
attributions = ig.attribute(
    inputs=input_ids,
    additional_forward_args=(attention_mask, sentiment_features),  # Pass sentiment features
    target=labels
)

# Convert attributions to numpy for Quantus
attributions_np = attributions.detach().cpu().numpy()

# Define Quantus metrics
metrics = {
    "Sparsity": quantus.Sparsity(),
    "Complexity": quantus.Complexity(),
    "Faithfulness": quantus.FaithfulnessCorrelation(),
    "Robustness": quantus.LocalLipschitzEstimate(),
    "Sensitivity": quantus.SensitivityN()
}

# Evaluate metrics
results = {}
for metric_name, metric in metrics.items():
    result = metric(
        model=wrapped_model,
        x_batch=input_ids.cpu().numpy(),
        y_batch=labels.cpu().numpy(),
        a_batch=attributions_np,
        explain_func=lambda x: attributions_np  # Use precomputed attributions
    )
    results[metric_name] = result

# Print results
for metric_name, result in results.items():
    print(f"{metric_name}: {result}")

# Export results to CSV
results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Score"])
results_df["Model"] = model_name_for_csv  # Add model name to the DataFrame
results_df.reset_index(inplace=True)
results_df.rename(columns={"index": "Metric"}, inplace=True)

# Save to CSV
csv_filename = f"quantus_metrics_{model_name_for_csv}.csv"
results_df.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

### Plot similarity between test set cases and the learned concepts similar to figure 3 in their paper

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

# Get prototype vectors (concepts)
prototypes = proto.prototypes.detach().cpu().numpy()  # shape: (num_prototypes, hidden_dim)

# Get representations for some samples (e.g., from your test set)
batch = next(iter(drug_review_dm.test_dataloader()))
with torch.no_grad():
    # Get last hidden states for the batch
    llm_out = proto.LLM(
        input_ids=batch['input_ids'].to(proto.device),
        attention_mask=batch['attention_mask'].to(proto.device),
        output_hidden_states=True
    )
    sample_reps = llm_out.hidden_states[-1][:, 0, :].cpu().numpy()  # [CLS] token or mean pooling



# prototypes: (num_prototypes, hidden_dim)
# sample_reps: (batch_size, hidden_dim)
# Assume proto.prototype_class_vec exists and is (num_prototypes, num_classes)

# 1. Identify positive and negative prototypes
# If proto.prototype_class_vec is one-hot or softmax over classes:
proto_class = proto.prototype_class_vec.detach().cpu().numpy()  # (num_prototypes, num_classes)
positive_proto_idx = np.argmax(proto_class[:, 1])  # class 1 = positive
negative_proto_idx = np.argmax(proto_class[:, 0])  # class 0 = negative

positive_proto = torch.tensor(prototypes[positive_proto_idx])
negative_proto = torch.tensor(prototypes[negative_proto_idx])

# 2. Compute similarities for each sample
sample_vecs = torch.tensor(sample_reps)  # (batch_size, hidden_dim)
sim_pos = F.cosine_similarity(sample_vecs, positive_proto.unsqueeze(0), dim=1)
sim_neg = F.cosine_similarity(sample_vecs, negative_proto.unsqueeze(0), dim=1)

# 3. Get ground-truth labels for the batch
labels = batch['labels'].cpu().numpy()

# 4. Plot
plt.figure(figsize=(8, 8))
for label in np.unique(labels):
    idxs = np.where(labels == label)[0]
    plt.scatter(sim_pos[idxs], sim_neg[idxs], label=f"Class {label}", alpha=0.7)
plt.xlabel("Similarity to Positive Prototype")
plt.ylabel("Similarity to Negative Prototype")
plt.title("2D Prototypical Space (like Proto-LM Fig. 3)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.nn.functional as F

sample_vec = sample_reps[0]  # pick one sample
proto_vecs = torch.tensor(prototypes)
similarities = F.cosine_similarity(torch.tensor(sample_vec).unsqueeze(0), proto_vecs)
plt.bar(range(len(similarities)), similarities.numpy())
plt.xlabel("Prototype Index")
plt.ylabel("Cosine Similarity")
plt.title("Sample-Prototype Similarity")
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F

# 1. Get all review texts and their embeddings from the training set
train_dataset = drug_review_dm.dataset["train"]
all_texts = train_dataset["review"]

# Get all input_ids and attention_mask for the train set
input_ids = train_dataset["input_ids"]
attention_mask = train_dataset["attention_mask"]

# Compute all embeddings (CLS token)
all_embeddings = []
batch_size = 128
for i in range(0, len(input_ids), batch_size):
    batch_input_ids = input_ids[i:i+batch_size].to(proto.device)
    batch_attention_mask = attention_mask[i:i+batch_size].to(proto.device)
    with torch.no_grad():
        outputs = proto.LLM(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            output_hidden_states=True
        )
        batch_embeds = outputs.hidden_states[-1][:, 0, :].cpu()  # CLS token
        all_embeddings.append(batch_embeds)
all_embeddings = torch.cat(all_embeddings, dim=0)  # (num_samples, hidden_dim)

# 2. For each prototype, find the closest text
prototypes = proto.prototypes.detach().cpu()  # (num_prototypes, hidden_dim)
closest_texts = []
for proto_vec in prototypes:
    sims = F.cosine_similarity(all_embeddings, proto_vec.unsqueeze(0), dim=1)
    idx = torch.argmax(sims).item()
    closest_texts.append(all_texts[idx])

# 3. Save to CSV
df = pd.DataFrame({"prototype_index": range(len(closest_texts)), "closest_text": closest_texts})
df.to_csv("prototype_texts.csv", index=False)